# 3 · Test: final numbers, then (optionally) ship

Scores the checkpoint chosen in **02_validate** on the **test** split — once. Don't go back and pick another
checkpoint because of these numbers, or the test split stops being an honest estimate.

```
datasets/rgb_image/test/<patch_info>/{0,1,2}/*.png
```
With `EXPORT = True` the checkpoint is copied into `resources/anti_spoof_models/`, where
`liveness-service predict` / `LivenessChecker` load it. **Every `.pth` there joins the ensemble**, so remove or
replace the model it supersedes.

## Parameters

In [ ]:
from model_development.config import REPO_ROOT

PATCH_INFO = "2.7_80x80"
CHECKPOINT = None   # file name printed by 02_validate, e.g. "2.7_2026-11-25-10-00-iter-900_80x80_MiniFASNetV2SE.pth"
SNAPSHOT_DIR = REPO_ROOT / "saved_logs" / "snapshot" / f"Anti_Spoofing_{PATCH_INFO}"
TEST_DIR = REPO_ROOT / "datasets" / "rgb_image" / "test" / PATCH_INFO
DEVICE = "cpu"
EXPORT = False      # copy into resources/anti_spoof_models after looking at the numbers

## Score on the test split

In [ ]:
from model_development.evaluation import evaluate

def show(results):
    """Print metrics as a table: lower APCER/BPCER/ACER is better."""
    print(f"{'checkpoint':58s} {'n':>5s} {'acc':>6s} {'APCER':>6s} {'BPCER':>6s} {'ACER':>6s}")
    for r in results:
        print(f"{r['checkpoint']:58s} {r['samples']:5d} {r['accuracy']:6.3f} {r['apcer']:6.3f} {r['bpcer']:6.3f} {r['acer']:6.3f}")

assert CHECKPOINT, "set CHECKPOINT to the best checkpoint from 02_validate"
result = evaluate(SNAPSHOT_DIR / CHECKPOINT, TEST_DIR, DEVICE)
show([result])
print("confusion matrix (rows: true 0/1/2, columns: predicted 0/1/2):")
for row in result["confusion_matrix"]:
    print("  ", row)

## Export

In [ ]:
import shutil

from liveness_service.ml.utils import parse_model_name

if EXPORT:
    height, width, model_type, _ = parse_model_name(CHECKPOINT)
    scale = PATCH_INFO.rsplit("_", 1)[0]
    target = REPO_ROOT / "resources" / "anti_spoof_models" / f"{scale}_{height}x{width}_{model_type}.pth"
    assert not target.exists(), f"{target.name} exists: remove it first if this model replaces it"
    shutil.copy2(SNAPSHOT_DIR / CHECKPOINT, target)
    print("exported ->", target)
else:
    print("EXPORT is False: nothing copied")